# 03 - Modeling
**Project**: Interpretable Student Academic Risk Prediction  
**Course**: CSE2267 – Machine Learning Techniques

This notebook trains multiple machine learning models to predict student academic risk and compares their performance.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, classification_report, confusion_matrix,
                             ConfusionMatrixDisplay, RocCurveDisplay)

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

sns.set(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)
print("Libraries imported successfully!")

## 1. Load Preprocessed Data

We try to load the preprocessed file first. If not available, we load the raw data and do basic preprocessing.

In [ ]:
import os

# Try loading preprocessed data
processed_path = "../data/processed/student_processed.csv"
raw_path = "../data/student-mat.csv"

if os.path.exists(processed_path):
    df = pd.read_csv(processed_path)
    print("Loaded preprocessed data from:", processed_path)
else:
    print("Preprocessed file not found. Loading raw data and applying basic preprocessing...")
    df = pd.read_csv(raw_path)
    
    # Create target
    df['At_Risk'] = (df['G3'] < 10).astype(int)
    
    # Drop grade-related columns to avoid data leakage (G1, G2, G3)
    cols_to_drop = ['G1', 'G2', 'G3']
    df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])
    
    # One-hot encode categorical columns
    cat_cols = df.select_dtypes(include=['object']).columns.tolist()
    df = pd.get_dummies(df, columns=cat_cols, drop_first=True)
    
    print("Basic preprocessing completed.")

print("Shape:", df.shape)
df.head()

In [ ]:
# Define features and target
TARGET = 'At_Risk'

if TARGET not in df.columns:
    raise ValueError(f"Target column '{TARGET}' not found in the dataframe!")

X = df.drop(columns=[TARGET])
y = df[TARGET]

print("Features shape:", X.shape)
print("Target distribution:")
print(y.value_counts(normalize=True).round(3) * 100)

## 2. Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape :", X_test.shape)
print("\nTrain target distribution:")
print(y_train.value_counts(normalize=True).round(3)*100)

## 3. Define Models

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1),
    "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42),
    "LightGBM": LGBMClassifier(random_state=42, verbose=-1)
}

print("Models defined:", list(models.keys()))

## 4. Train & Evaluate Models

In [ ]:
results = []
trained_models = {}

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for name, model in models.items():
    print(f"\n{'='*50}")
    print(f"Training: {name}")
    print('='*50)
    
    # Train
    model.fit(X_train, y_train)
    trained_models[name] = model
    
    # Predictions
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1] if hasattr(model, "predict_proba") else None
    
    # Metrics
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba) if y_proba is not None else np.nan
    
    # Cross-validation F1
    cv_f1 = cross_val_score(model, X, y, cv=skf, scoring='f1').mean()
    
    results.append({
        "Model": name,
        "Accuracy": round(acc, 4),
        "Precision": round(prec, 4),
        "Recall": round(rec, 4),
        "F1-Score": round(f1, 4),
        "ROC-AUC": round(auc, 4),
        "CV F1 (5-fold)": round(cv_f1, 4)
    })
    
    print(f"Accuracy : {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall   : {rec:.4f}")
    print(f"F1-Score : {f1:.4f}")
    print(f"ROC-AUC  : {auc:.4f}")
    print(f"CV F1    : {cv_f1:.4f}")
    
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, target_names=['Not At-Risk', 'At-Risk']))
    
    # Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Not At-Risk', 'At-Risk'])
    disp.plot(cmap='Blues')
    plt.title(f"Confusion Matrix - {name}")
    plt.tight_layout()
    plt.savefig(f"../results/figures/cm_{name.replace(' ', '_').lower()}.png", dpi=150, bbox_inches='tight')
    plt.show()

## 5. Model Comparison

In [ ]:
results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by="F1-Score", ascending=False).reset_index(drop=True)

print("\n===== Model Comparison =====")
display(results_df)

# Save comparison table
results_df.to_csv("../results/model_comparison.csv", index=False)
print("\nComparison table saved to results/model_comparison.csv")

In [ ]:
# Bar plot of F1-Scores
plt.figure(figsize=(10, 6))
sns.barplot(data=results_df, x="F1-Score", y="Model", palette="viridis")
plt.title("Model Comparison - F1 Score")
plt.xlim(0, 1)
plt.tight_layout()
plt.savefig("../results/figures/model_comparison_f1.png", dpi=150, bbox_inches='tight')
plt.show()

## 6. Save Best Model

In [ ]:
best_model_name = results_df.iloc[0]["Model"]
best_model = trained_models[best_model_name]

print(f"Best Model: {best_model_name}")
print(f"F1-Score  : {results_df.iloc[0]['F1-Score']}")
print(f"ROC-AUC   : {results_df.iloc[0]['ROC-AUC']}")

# Save model
os.makedirs("../results/models", exist_ok=True)
model_path = f"../results/models/best_model_{best_model_name.replace(' ', '_').lower()}.joblib"
joblib.dump(best_model, model_path)
print(f"\nBest model saved to: {model_path}")

# Also save feature names for later use in SHAP
joblib.dump(list(X.columns), "../results/models/feature_names.joblib")
print("Feature names saved.")

## Summary

- Multiple models trained and evaluated
- Best model selected based on F1-Score
- Confusion matrices and comparison plots saved in `results/figures/`
- Best model saved in `results/models/`

**Next**: Run `04_SHAP_Explainability.ipynb` for interpretability analysis.